<a href="https://colab.research.google.com/github/amranbelete/DS2002FA26/blob/main/2026_09_25_cleaning_gauntlet_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [ ]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [ ]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [ ]:
# TODO
n_before = len(df)
dupe_count = df.duplicated().sum()
print(f"Found {dupe_count}  duplicate rows")

df = df.drop_duplicates().reset_index(drop=True)

log("drop_duplicates", "removed the  duplicate rows", n_before - len(df))

Found 0  duplicate rows
[drop_duplicates] removed the  duplicate rows (0 row(s))


### TODO 2 — clean `price` -> float

In [ ]:
# TODO
df['price'] = (
    df['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
    .astype(float)
)

log("clean_price", "took '$' out and cast price to float", len(df))

[clean_price] took '$' out and cast price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [ ]:
# TODO
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

poor_qty_mask = df['qty'].isna() | (df['qty'] < 0)
n_poor = poor_qty_mask.sum()

df = df[~poor_qty_mask].reset_index(drop=True)

log("clean_qty", "dropped the rows with missing or negative qty", n_poor)

[clean_qty] dropped the rows with missing or negative qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [ ]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse

print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
}


n_before_map = (df['item'] != df['item'].map(ITEM_MAP)).sum()
df['item'] = df['item'].map(ITEM_MAP)

log("canonicalize_item", "turned 6 spelling variants into 3 item names", n_before_map)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[canonicalize_item] turned 6 spelling variants into 3 item names (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [ ]:
# TODO
print(df['category'].value_counts())

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'RainGear': 'RainGear',
    'rain-gear': 'RainGear',
    'Merch': 'Merch',
    'Apparel': 'Apparel',
}

n_changed = (df['category'] != df['category'].map(CATEGORY_MAP)).sum()
df['category'] = df['category'].map(CATEGORY_MAP)

log(
    "normalize_category",
    "merged Food/food and RainGear/rain-gear as spelling variants. "
    "kept Merch and Apparel as separate categories (business decision: "
    "apparel is a distinct product type from general merch).",
    n_changed
)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[normalize_category] Merged Food/food and RainGear/rain-gear as spelling variants. Kept Merch and Apparel as SEPARATE categories (business decision: apparel is a distinct product type from general merch, not a spelling issue). (89 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [ ]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float

#TODO: assert something about item

assert df['item'].nunique() == 3
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
#TODO: assert something about category

assert df['category'].nunique() == 4
assert set(df['category'].unique()) == {'Food', 'RainGear', 'Merch', 'Apparel'}


print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [ ]:
# TODO

df['revenue'] = df['qty'] * df['price']

revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
print(revenue_by_category)

total_revenue = round(df['revenue'].sum(), 2)
print(f"\nTotal revenue: {total_revenue}")

category
Food        1656.0
RainGear    1512.0
Merch        856.5
Apparel      715.5
Name: revenue, dtype: float64

Total revenue: 4740.0


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [ ]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,drop_duplicates,Removed exact duplicate rows,15
1,drop_duplicates,removed the duplicate rows,0
2,clean_price,Stripped '$' and cast price to float,300
3,clean_price,took '$' out and cast price to float,300
4,clean_qty,dropped the rows with missing or negative qty,25
5,canonicalize_item,turned 6 spelling variants into 3 item names,126
6,normalize_category,Merged Food/food and RainGear/rain-gear as spe...,89


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

the step that changed the revenue total the most was the 'dropping duplicates'. It went from 4,852.50 to 4,594.50, which is a swinfg of $258.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

I think for the duplicate drop step, i treated the 15 repeated rows as data generation artifacs and removed them, however i think a reasonable person could argue that they could they represent legitimate repeat orders.


_your answer here_